In [ ]:
import torch
import numpy as np
np.set_printoptions(precision=2, linewidth=140)
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.__version__


In [ ]:
!rm -rf kernelforge/
!git clone https://github.com/marcalph/kernelforge.git

In [ ]:
# wurlitzer -> capture c level stdout/err, ninja -> cpp build tool
!pip install wurlitzer ninja torchvision
%load_ext wurlitzer

# Grayscale

### python version

In [ ]:
import torchvision.transforms.functional as tvf
from torchvision import io
from kernelforge.src.py.utils import show_img

def rgb2gray_py(x):
    c,h,w = x.shape
    n = h*w
    x = x.flatten()
    res = torch.empty(n, dtype=x.dtype, device=x.device)
    for i in range(n): res[i] = 0.2989*x[i] + 0.5870*x[i+n] + 0.1140*x[i+2*n]
    return res.view(h,w)
     


img = io.read_image('./kernelforge/pisco.jpg') # CHW
img_small = tvf.resize(img, 150, antialias=True)
ch,h,w = img_small.shape
show_img(img_small)



In [ ]:
%%time
img_g = rgb2gray_py(img_small)
     


In [ ]:
show_img(img_g, cmap='gray')
     


### CUDA
!rm -rf /root/.cache/torch_extensions/py312_cu128/

In [ ]:
!rm -rf /root/.cache/torch_extensions/py312_cu128/

In [ ]:
from pathlib import Path
from kernelforge.src.py.compile import compile_extension, get_sig
from torchvision.io import read_image

def main(ext):
    """
    Read input image, convert it to grayscale via custom CUDA kernel and write out as png.
    """
    x = read_image("kernelforge/pisco.jpg").permute(1, 2, 0).cuda()
    print("Input image:", x.shape, x.dtype, "mean:", x.float().mean().item())

    assert x.dtype == torch.uint8

    y = ext.grayscale(x)

    print("Output image:", y.shape, y.dtype, "mean:", y.float().mean().item())
    return y.cpu()

module = compile_extension("kernelforge/src/kernels/grayscale.cu", )
res = main(module)


In [ ]:
x = read_image("kernelforge/pisco.jpg")
x.shape

In [ ]:
imgc = img.permute(1,2,0).contiguous().cuda()
imgc.shape


In [ ]:
%time  res = module.grayscale(imgc); torch.cuda.synchronize()
# bad estimation we pay the CUDA driver dispatch + context switching
res.shape

In [ ]:
n = 2048
t = torch.randint(0, 256, (n, n, 3), dtype=torch.uint8, device="cuda")
out = module.grayscale(t); torch.cuda.synchronize()
times=10000
import time
t0 = time.perf_counter_ns()
for i in range(times):
    module.grayscale(t)
torch.cuda.synchronize()
t1 = time.perf_counter_ns()

print((t1-t0) / times / 1_000, "µs") 

In [ ]:
%%time
for i in range(10_000):
    module.grayscale(t)
torch.cuda.synchronize()

In [ ]:
show_img(res, cmap='gray')

In [ ]:
with torch.profiler.profile() as prof:
    for i in range(10_000):
        module.grayscale(t)
        torch.cuda.synchronize()

print(prof.key_averages().table())

# gelu non linearity / fusion

In [ ]:
# as per the pytorch doc
def gelu(x):
    return 0.5 * x * (1+ torch.tanh((2/torch.pi)**0.5 * (x+0.044715 * x**3)))

x = torch.randn(1024, 1024, device="cuda")
(torch.nn.functional.gelu(x, approximate='tanh') - gelu(x)).abs().max()

In [ ]:
%timeit g1 = gelu(x); torch.cuda.synchronize()
%timeit g2 = torch.nn.functional.gelu(x, approximate='tanh'); torch.cuda.synchronize()



In [ ]:
with torch.profiler.profile() as prof:
    %timeit -n 1000 gelu(x)

print(prof.key_averages().table())
# slower because to many kernel launch

# Blur

In [ ]:

module = compile_extension(
    cuda_source = "kernelforge/src/kernels/blur.cu",
    cpp_source = "torch::Tensor blur(torch::Tensor image, int radius);"
    )
def main(ext):
    """
    Read input image, convert it to grayscale via custom CUDA kernel and write out as png.
    """
    x = read_image("kernelforge/pisco.jpg").permute(1, 2, 0).cuda()
    print("Input image:", x.shape, x.dtype, "mean:", x.float().mean().item())

    assert x.dtype == torch.uint8

    y = ext.blur(x, 8)

    print("Output image:", y.shape, y.dtype, "mean:", y.float().mean().item())
    return y.cpu()


blurred = main(module)


In [ ]:
%time
res = module.blur(imgc, 12).cpu()
res.shape

In [ ]:

show_img(blurred.permute(2,0,1))

# matmul

In [ ]:
from kernelforge.src.py.compile import compile_extension

module = compile_extension(
    "kernelforge/src/kernels/matmul.cu",
    funcs=["matmul"]
    )



In [ ]:
import torch
a, b = torch.randn(5120, 256), torch.randn(256, 5120)
ac, bc = a.contiguous().cuda(), b.contiguous().cuda()


In [ ]:
%%time
tr = (ac@bc).cpu()


In [ ]:
%%time
res = module.matmul(ac, bc).cpu()


In [ ]:
torch.isclose(tr, (ac @ bc).cpu(), atol=1e-5).all()

In [ ]:
%%timeit -n 10
module.matmul(ac,bc)
torch.cuda.synchronize()
     
